# Tutorial: Belgium vs Netherlands Walk-Forward

Audience:
- Quants, developers, and asset analysts comparing the same BESS decision process across European markets.

Learning goals:
- Run the same `schema_version: 4` config shape in Belgium and the Netherlands.
- Inspect market-level oracle gaps and settlement differences.
- Reuse the shared comparison layer across market adapters.


## Outline

1. Load the v0.5 single-asset configs
2. Run Belgium and Netherlands walk-forward benchmarks
3. Compare normalized KPIs and artifact summaries


In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd

from euroflex_bess_lab import compare_runs, load_config, run_walk_forward
from euroflex_bess_lab.analytics.reporting import load_report_summary

REPO_ROOT = Path.cwd()
EXAMPLE_CONFIG_DIR = REPO_ROOT / "examples" / "configs"
ARTIFACT_ROOT = Path(tempfile.mkdtemp(prefix="euroflex-v10-cross-market-"))
ARTIFACT_ROOT

## Step 1 - Load Belgium and Netherlands configs

Both configs use the same battery shape and the same generic `workflow` field. The only meaningful differences are the `market.id`, timezone, and imbalance settlement data.


In [ ]:
belgium_config = load_config(EXAMPLE_CONFIG_DIR / "canonical" / "belgium_full_stack.yaml")
netherlands_config = load_config(EXAMPLE_CONFIG_DIR / "basic" / "netherlands_da_only_base.yaml")

belgium_config.artifacts.root_dir = ARTIFACT_ROOT / "belgium"
netherlands_config.artifacts.root_dir = ARTIFACT_ROOT / "netherlands"

{
    "belgium": {
        "workflow": belgium_config.workflow,
        "base_workflow": belgium_config.execution_workflow,
        "market": belgium_config.market.id,
    },
    "netherlands": {
        "workflow": netherlands_config.workflow,
        "base_workflow": netherlands_config.execution_workflow,
        "market": netherlands_config.market.id,
    },
}

## Step 2 - Run the walk-forward engine

The same `run_walk_forward()` entrypoint now dispatches through a market registry. Belgium settles imbalance with a single realized price; Netherlands uses a dual-price shortage/surplus settlement basis.


In [ ]:
belgium_result = run_walk_forward(belgium_config)
netherlands_result = run_walk_forward(netherlands_config)

summary_frame = pd.DataFrame(
    [
        load_report_summary(belgium_result.output_dir),
        load_report_summary(netherlands_result.output_dir),
    ]
)[
    [
        "market_id",
        "workflow",
        "benchmark_name",
        "settlement_basis",
        "total_pnl_eur",
        "oracle_gap_total_pnl_eur",
        "throughput_mwh",
    ]
]
summary_frame

## Step 3 - Compare runs with the shared comparison layer

The compare command and Python helper now understand mixed-market runs and can group outputs by market without any market-specific branching.


In [ ]:
comparison_dir = compare_runs(
    [belgium_result.output_dir, netherlands_result.output_dir],
    ARTIFACT_ROOT / "comparison",
    group_by="market",
)

comparison_frame = pd.read_csv(comparison_dir / "comparison.csv")
grouped_frame = pd.read_csv(comparison_dir / "grouped_by_market.csv")
comparison_frame[["market_id", "benchmark_name", "total_pnl_eur", "oracle_gap_total_pnl_eur"]], grouped_frame

## Step 4 - Inspect adapter-specific dispatch context

The stable artifact contract stays the same, but the Netherlands dispatch includes extra dual-price fields that Belgium does not need.


In [ ]:
dispatch_columns = {
    "belgium": [c for c in belgium_result.realized_dispatch.columns if "imbalance" in c][:6],
    "netherlands": [c for c in netherlands_result.realized_dispatch.columns if "imbalance" in c][:8],
}
dispatch_columns

## Takeaways

- The repo is now a market-plugin framework rather than a Belgium-only benchmark.
- `summary.json`, `decision_log.parquet`, `settlement_breakdown.parquet`, and `forecast_snapshots.parquet` carry stable market metadata.
- Private forecasts or future reserve products can plug into the same walk-forward and comparison chassis.
